<a href="https://colab.research.google.com/github/Carinaaa/ML-Learning-Path/blob/intro-LLM/RAG_Intro_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-chroma

In [2]:
!pip install langchain-openai

In [3]:
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [4]:
import os
import glob
import gradio as gr
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from google.colab import userdata
import requests

In [5]:
MODEL = 'gpt-4o-mini'
db_name = 'vector_db'

In [6]:
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key # set it as an env var

## Retriving the documents

In [7]:
context = {}

# Employees
all_employees = ['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Avery Lancaster', 'Emily Carter', 'Emily Tran', 'Jordan Blake', 'Jordan K. Bishop', 'Maxine Thompson', 'Oliver Spencer', 'Samantha Greene', 'Samuel Trenton']
for e in all_employees:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/employees/{e.replace(" ", "%20")}.md').text

employee_context = context.copy()
try:
  os.mkdir('knowledge-base/')
  os.mkdir('knowledge-base/employees')
except FileExistsError:
  print("Dirs already exists.")
for e in all_employees:
  with open(f'knowledge-base/employees/{e}.md', 'w') as f:
    f.write(employee_context[e])

# Products
all_products = ['Carllm', 'Homellm', 'Markellm', 'Rellm']
products_context = {}
for p in all_products:
  context[p] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/products/{p}.md').text
  products_context[p] = context[p]

try:
  os.mkdir('knowledge-base/products')
except FileExistsError:
  print("Dirs already exists.")
for p in all_products:
  with open(f'knowledge-base/products/{p}.md', 'w') as f:
    f.write(products_context[p])

# Company info
company_info = ['about', 'careers', 'overview']
company_context = {}
for e in company_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/company/{e}.md').text
  company_context[e] = context[e]
try:
  os.mkdir('knowledge-base/company')
except FileExistsError:
  print("Dirs already exists.")
for e in company_info:
  with open(f'knowledge-base/company/{e}.md', 'w') as f:
    f.write(company_context[e])

# Contracts
contracts_info = ['Contract with Apex Reinsurance for Rellm', 'Contract with Belvedere Insurance for Markellm', 'Contract with BrightWay Solutions for Markellm',
                'Contract with EverGuard Insurance for Rellm', 'Contract with GreenField Holdings for Markellm', 'Contract with GreenValley Insurance for Homellm',
                'Contract with Greenstone Insurance for Homellm','Contract with Pinnacle Insurance Co. for Homellm', 'Contract with Roadway Insurance Inc. for Carllm',
                'Contract with Stellar Insurance Co. for Rellm', 'Contract with TechDrive Insurance for Carllm', 'Contract with Velocity Auto Solutions for Carllm']
contracts_context = {}
for e in contracts_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/contracts/{e.replace(" ", "%20")}.md').text
  contracts_context[e] = context[e]
try:
  os.mkdir('knowledge-base/contracts')
except FileExistsError:
  print("Dirs already exists.")
for e in contracts_info:
  with open(f'knowledge-base/contracts/{e}.md', 'w') as f:
    f.write(contracts_context[e])

Dirs already exists.
Dirs already exists.
Dirs already exists.
Dirs already exists.


In [8]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

folders = glob.glob('knowledge-base/*')
# Specify encoding in header
text_loader_kwargs = {'encoding': 'utf-8'}

documents = []
for folder in folders:
  doc_type = os.path.basename(folder)
  loader = DirectoryLoader(folder, glob='**/*.md', loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
  folder_docs = loader.load()
  print(folder_docs)
  for doc in folder_docs:
    doc.metadata['doc_type'] = doc_type
    documents.append(doc)

[Document(metadata={'source': 'knowledge-base/company/careers.md'}, page_content='# Careers at Insurellm\n\nInsurellm is hiring! We are looking for talented software engineers, data scientists and account executives to join our growing team. Come be a part of our movement to disrupt the insurance sector.'), Document(metadata={'source': 'knowledge-base/company/overview.md'}, page_content='# Overview of Insurellm\n\nInsurellm is an innovative insurance tech firm with 200 employees across the US.\nInsurellm offers 4 insurance software products:\n- Carllm, a portal for auto insurance companies\n- Homellm, a portal for home insurance companies\n- Rellm, an enterprise platform for the reinsurance sector\n- Marketllm, a marketplace for connecting consumers with insurance providers\n  \nInsurellm has more than 300 clients worldwide.'), Document(metadata={'source': 'knowledge-base/company/about.md'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insur

In [9]:
documents

[Document(metadata={'source': 'knowledge-base/company/careers.md', 'doc_type': 'company'}, page_content='# Careers at Insurellm\n\nInsurellm is hiring! We are looking for talented software engineers, data scientists and account executives to join our growing team. Come be a part of our movement to disrupt the insurance sector.'),
 Document(metadata={'source': 'knowledge-base/company/overview.md', 'doc_type': 'company'}, page_content='# Overview of Insurellm\n\nInsurellm is an innovative insurance tech firm with 200 employees across the US.\nInsurellm offers 4 insurance software products:\n- Carllm, a portal for auto insurance companies\n- Homellm, a portal for home insurance companies\n- Rellm, an enterprise platform for the reinsurance sector\n- Marketllm, a marketplace for connecting consumers with insurance providers\n  \nInsurellm has more than 300 clients worldwide.'),
 Document(metadata={'source': 'knowledge-base/company/about.md', 'doc_type': 'company'}, page_content="# About In

## Create chunks

In [10]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [11]:
len(chunks)

123

In [12]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f'Document types found: {", ".join(doc_types)}')

Document types found: products, company, employees, contracts


## Auto-Encoding LLMs

*   Mapping each chunk of text into a Vector that represents the meaning of th text, known as an embedding
*   auto-encoding LLMs generates outputs given a complete input (often used in classification)
*   auto-regressive LLMs generate tokens only based on past context

Note: there are other solutions to create vectors such that the db remains internal

### *Details*

**Auto-encoding LLMs**
*   process entire sequences simultaneously in both directions
*   use bidirectional attention - each token can attend to all other tokens in the sequence
*   trained using objectives like masked language modeling (predicting randomly masked tokens)
*   **Excel at understading and representing text rather then generating it**
*   Can naturally handle tasks requiring full context understanding
*   Better for classification, question answering, andother discriminative tasks

**Auto-regressive LLMs**
*   generates text sequentially, one token at a time from left to right
*   At each step, they predict the next token based on all previous tokens
*   Use causal masing during training - each position can only 'see' previous positions
*   **Well-suited fo text generation tasks**
*   Training objective: maximize the pobabiity of the next token given previuos context
*   Cannot directly "fill in the blanks" in the middle of the sequences

In [13]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
embeddings = OpenAIEmbeddings()

# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [14]:
# Check if a Chroma Datastore already exists - if so, delete the collection to start from scratch

if os.path.exists(db_name):
  Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collenction()

In [16]:
# Create our Chroma vectorstore!

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f'Vectorstore created with {vectorstore._collection.count()} documents')

Vectorstore created with 246 documents


In [17]:
# Get one vector and find how many dimensions it has

collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=['embeddings'])['embeddings'][0]
dimensions = len(sample_embedding)
print(f'The vectors have {dimensions:,} dimensions')

The vectors have 1,536 dimensions


In [19]:
another_sample_embedding=collection.get(limit=1)
another_sample_embedding

{'ids': ['0fe7b083-68d1-4ca5-a5ea-b76029e45084'],
 'embeddings': None,
 'documents': ['# Careers at Insurellm\n\nInsurellm is hiring! We are looking for talented software engineers, data scientists and account executives to join our growing team. Come be a part of our movement to disrupt the insurance sector.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'knowledge-base/company/careers.md',
   'doc_type': 'company'}]}

In [20]:
another_sample_embedding=collection.get(limit=1, include=['embeddings'])
another_sample_embedding

{'ids': ['0fe7b083-68d1-4ca5-a5ea-b76029e45084'],
 'embeddings': array([[-0.02090848, -0.02410781,  0.00598381, ...,  0.00891431,
         -0.00978384, -0.01904995]]),
 'documents': None,
 'uris': None,
 'included': ['embeddings'],
 'data': None,
 'metadatas': None}

In [22]:
another_sample_embedding=collection.get(limit=1, include=['documents'])
another_sample_embedding

{'ids': ['0fe7b083-68d1-4ca5-a5ea-b76029e45084'],
 'embeddings': None,
 'documents': ['# Careers at Insurellm\n\nInsurellm is hiring! We are looking for talented software engineers, data scientists and account executives to join our growing team. Come be a part of our movement to disrupt the insurance sector.'],
 'uris': None,
 'included': ['documents'],
 'data': None,
 'metadatas': None}

In [23]:
another_sample_embedding=collection.get(limit=1, include=['metadatas'])
another_sample_embedding

{'ids': ['0fe7b083-68d1-4ca5-a5ea-b76029e45084'],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas'],
 'data': None,
 'metadatas': [{'doc_type': 'company',
   'source': 'knowledge-base/company/careers.md'}]}

## Visualizing the Vector Store

In [26]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
doc_types = [metadata['doc_type'] for metadata in result['metadatas']]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees','contracts','company'].index(t)] for t in doc_types]

In [27]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:,0],
    y=reduced_vectors[:,1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f'Type: {t}<br>Text: {d[:100]}...' for t, d in zip(doc_types,documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [28]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()